> **About this notebook**
>
> This notebook is part of the example gallery of [CLABTOOLKIT](https://github.com/connectomicslab/clabtoolkit), an open-source Python toolkit for routine processing, manipulation and visualization of neuroimaging data. It shows how to use the `connectivitytools` module.
>
> 📦 **Install:** `pip install clabtoolkit` ([PyPI](https://pypi.org/project/clabtoolkit/)) · 📖 **Docs:** [clabtoolkit.readthedocs.io](https://clabtoolkit.readthedocs.io) · 📄 **Paper:** [arXiv:2607.02638](https://arxiv.org/abs/2607.02638)
>
> **Author:** Yasser Alemán-Gómez (Connectomics Lab, Radiology Department, CHUV – Lausanne University Hospital)

## Testing Methods in the `PLOTTOOLS` Module

This Jupyter Notebook serves as a platform to rigorously test the various methods included in the Python module `PLOTTOOLS`.

This module is particularly valuable for researchers preparing figures that must adapt to the display
being used: querying the screen and monitor properties, computing optimal subplot grids and layouts
proportional to the screen, scaling the font sizes for matplotlib and PyVista renderings, and sampling
spherical coordinates for layout purposes.

The tests are organized into the following sections for clarity and ease of navigation:

### Table of Contents

---

* 🔷 1. [`Screen and Monitor Information`](#plot-screen). Methods to get the properties of the screen and the monitors
    * 🔸 1.1. [`get_screen_size`](#plot-get_screen_size). Get the current screen size in pixels
    * 🔸 1.2. [`get_current_monitor_size`](#plot-get_current_monitor_size). Get the size of the monitor where the mouse cursor is located
    * 🔸 1.3. [`estimate_monitor_dpi`](#plot-estimate_monitor_dpi). Estimate the monitor DPI based on the screen resolution using common monitor configurations

* 🔷 2. [`Subplot Layouts`](#plot-layouts). Methods to compute optimal subplot grids and layouts
    * 🔸 2.1. [`calculate_optimal_subplots_grid`](#plot-calculate_optimal_subplots_grid). Calculate the optimal grid dimensions for a given number of views
    * 🔸 2.2. [`calculate_subplot_layout`](#plot-calculate_subplot_layout). Calculate the optimal rows and columns for subplots based on the screen proportions
    * 🔸 2.3. [`create_proportional_subplots`](#plot-create_proportional_subplots). Create a figure with subplots arranged according to the screen proportions

* 🔷 3. [`Font Sizes`](#plot-fonts). Methods to compute the font sizes adapted to the figure dimensions and the monitor
    * 🔸 3.1. [`calculate_font_sizes`](#plot-calculate_font_sizes). Calculate appropriate font sizes for matplotlib plots based on the dimensions, the monitor DPI and the colorbar configuration
    * 🔸 3.2. [`get_pyvista_fonts`](#plot-get_pyvista_fonts). Get integer font sizes specifically for PyVista compatibility

* 🔷 4. [`Other Methods`](#plot-other_methods). Other auxiliary methods used to build the plots
    * 🔸 4.1. [`generate_spherical_coords`](#plot-generate_spherical_coords). Sample n_regions points uniformly inside a sphere of the given radius

---

---
## <a id="plot-screen"></a>1. Screen and Monitor Information
Methods to get the properties of the screen and the monitors.

### 1.1 `get_screen_size`
<a id="plot-get_screen_size"></a>

Get the current screen size in pixels.

In [ ]:
########################### Testing get_screen_size ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Getting the size of the screen in pixels...")
print("NOTE: It requires a graphical display (tkinter). It will fail on headless machines.")
try:
    width, height = cltplot.get_screen_size()
    print(f"Screen size: {width} x {height} pixels")
except Exception as e:
    width, height = 1920, 1080
    print(f"No display available ({e}). Using a fallback size of {width} x {height} pixels")
print("-------------------------------")
print(" ")

print("Test 2: Using the screen size to compute the aspect ratio of the screen...")
print(f"Aspect ratio (width/height): {width / height:.3f}")
print("-------------------------------")

### 1.2 `get_current_monitor_size`
<a id="plot-get_current_monitor_size"></a>

Get the size of the monitor where the mouse cursor is located.

In [ ]:
########################### Testing get_current_monitor_size ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Getting the size of the monitor where the mouse cursor is located...")
print("NOTE: It requires a graphical display (tkinter) and the screeninfo package.")
try:
    mon_width, mon_height = cltplot.get_current_monitor_size()
    print(f"Current monitor size: {mon_width} x {mon_height} pixels")
except Exception as e:
    mon_width, mon_height = 1920, 1080
    print(f"No display available ({e}). Using a fallback size of {mon_width} x {mon_height} pixels")
print("-------------------------------")
print(" ")

print("Test 2: Comparing the current monitor with the screen reported by get_screen_size...")
print("NOTE: In multi-monitor setups both sizes can be different.")
try:
    scr_width, scr_height = cltplot.get_screen_size()
    print(f"Screen size          : {scr_width} x {scr_height} pixels")
    print(f"Current monitor size : {mon_width} x {mon_height} pixels")
    print(f"Same size            : {(scr_width, scr_height) == (mon_width, mon_height)}")
except Exception as e:
    print(f"No display available ({e}).")
print("-------------------------------")

### 1.3 `estimate_monitor_dpi`
<a id="plot-estimate_monitor_dpi"></a>

Estimate the monitor DPI based on the screen resolution using common monitor configurations.

In [ ]:
########################### Testing estimate_monitor_dpi ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Estimating the DPI for common monitor resolutions...")
for res in [(1366, 768), (1920, 1080), (2560, 1440), (3840, 2160)]:
    dpi = cltplot.estimate_monitor_dpi(*res)
    print(f"  {res[0]} x {res[1]} -> {dpi} DPI")
print("-------------------------------")
print(" ")

print("Test 2: Estimating the DPI for non-standard resolutions (estimated from the pixel density)...")
for res in [(1280, 1024), (2048, 1536), (5120, 2880)]:
    dpi = cltplot.estimate_monitor_dpi(*res)
    print(f"  {res[0]} x {res[1]} -> {dpi} DPI")
print("-------------------------------")
print(" ")

print("Test 3: Estimating the DPI of the current monitor...")
try:
    width, height = cltplot.get_current_monitor_size()
except Exception:
    width, height = 1920, 1080
print(f"  {width} x {height} -> {cltplot.estimate_monitor_dpi(width, height)} DPI")
print("-------------------------------")

---
## <a id="plot-layouts"></a>2. Subplot Layouts
Methods to compute optimal subplot grids and layouts.

### 2.1 `calculate_optimal_subplots_grid`
<a id="plot-calculate_optimal_subplots_grid"></a>

Calculate the optimal grid dimensions for a given number of views.

In [ ]:
########################### Testing calculate_optimal_subplots_grid ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Computing the grid for a small number of views (1 to 8)...")
print("NOTE: The method returns the grid size [rows, columns] and the position of each view.")
for n_views in [1, 2, 3, 4, 6, 8]:
    grid_size, positions = cltplot.calculate_optimal_subplots_grid(n_views)
    print(f"  {n_views} views -> grid {grid_size}")
print("-------------------------------")
print(" ")

print("Test 2: Inspecting the position (row, column) assigned to each view...")
grid_size, positions = cltplot.calculate_optimal_subplots_grid(5)
print(f"  5 views -> grid {grid_size}")
for i, pos in enumerate(positions):
    print(f"    View {i}: {pos}")
print("-------------------------------")
print(" ")

print("Test 3: Computing the grid for 16 views (fixed 4 x 4 grid)...")
print("NOTE: For more than 8 views (except 16) the grid follows the screen proportions and requires a display.")
grid_size, positions = cltplot.calculate_optimal_subplots_grid(16)
print(f"  16 views -> grid {grid_size}")
print("-------------------------------")

### 2.2 `calculate_subplot_layout`
<a id="plot-calculate_subplot_layout"></a>

Calculate the optimal rows and columns for subplots based on the screen proportions.

In [ ]:
########################### Testing calculate_subplot_layout ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Computing the layout for 10 subplots on a Full HD screen (1920 x 1080)...")
rows, cols, aspect = cltplot.calculate_subplot_layout(10, screen_width=1920, screen_height=1080)
print(f"  Rows: {rows}, Columns: {cols}, Aspect ratio used: {aspect:.3f}")
print("-------------------------------")
print(" ")

print("Test 2: Computing the layout for 10 subplots on an ultrawide screen (3440 x 1440)...")
rows, cols, aspect = cltplot.calculate_subplot_layout(10, screen_width=3440, screen_height=1440)
print(f"  Rows: {rows}, Columns: {cols}, Aspect ratio used: {aspect:.3f}")
print("-------------------------------")
print(" ")

print("Test 3: Forcing a target aspect ratio (it overrides the screen dimensions)...")
for target in [0.5, 1.0, 2.0]:
    rows, cols, aspect = cltplot.calculate_subplot_layout(12, target_aspect_ratio=target)
    print(f"  12 subplots, target aspect {target} -> Rows: {rows}, Columns: {cols}")
print("-------------------------------")

### 2.3 `create_proportional_subplots`
<a id="plot-create_proportional_subplots"></a>

Create a figure with subplots arranged according to the screen proportions.

In [ ]:
########################### Testing create_proportional_subplots ###########################
import clabtoolkit.plottools as cltplot
import matplotlib.pyplot as plt
import numpy as np

print("Test 1: Creating a figure with 7 subplots following the proportions of a Full HD screen...")
print("NOTE: The unused subplots are hidden.")
fig, axes, layout_info = cltplot.create_proportional_subplots(
    7, figsize_base=3, screen_width=1920, screen_height=1080
)
print(f"  Layout information: {layout_info}")
x = np.linspace(0, 2 * np.pi, 100)
for i in range(7):
    axes[i].plot(x, np.sin((i + 1) * x))
    axes[i].set_title(f"sin({i + 1}x)")
plt.tight_layout()
plt.show()
print("-------------------------------")
print(" ")

print("Test 2: Creating a figure with 4 subplots using a square target aspect ratio...")
fig, axes, layout_info = cltplot.create_proportional_subplots(4, figsize_base=3, target_aspect_ratio=1.0)
print(f"  Layout information: {layout_info}")
rng = np.random.default_rng(seed=42)
for i, ax in enumerate(axes):
    ax.imshow(rng.random((20, 20)), cmap="viridis")
    ax.set_title(f"Random image {i + 1}")
    ax.axis("off")
plt.tight_layout()
plt.show()
print("-------------------------------")

---
## <a id="plot-fonts"></a>3. Font Sizes
Methods to compute the font sizes adapted to the figure dimensions and the monitor.

### 3.1 `calculate_font_sizes`
<a id="plot-calculate_font_sizes"></a>

Calculate appropriate font sizes for matplotlib plots based on the dimensions, the monitor DPI and the colorbar configuration.

In [ ]:
########################### Testing calculate_font_sizes ###########################
import clabtoolkit.plottools as cltplot
import matplotlib.pyplot as plt
import numpy as np

print("Test 1: Computing the font sizes for a 6 x 4 inches plot on a Full HD screen...")
fonts = cltplot.calculate_font_sizes(6, 4, screen_width=1920, screen_height=1080)
for key, value in fonts.items():
    print(f"  {key:15s}: {value}")
print("-------------------------------")
print(" ")

print("Test 2: Comparing the font sizes for different plot sizes and colorbar orientations on a 4K screen...")
for width, height, orientation in [(3, 2, "vertical"), (3, 2, "horizontal"), (12, 8, "vertical")]:
    fonts = cltplot.calculate_font_sizes(
        width, height, screen_width=3840, screen_height=2160, colorbar_orientation=orientation
    )
    print(
        f"  {width} x {height} in, {orientation:10s} -> title: {fonts['title']}, "
        f"ticks: {fonts['tick_labels']}, colorbar title: {fonts['colorbar_title']}"
    )
print("-------------------------------")
print(" ")

print("Test 3: Applying the computed font sizes to a matplotlib figure with a colorbar...")
fonts = cltplot.calculate_font_sizes(6, 4, colorbar_orientation="horizontal", auto_detect_monitor=False)
fig, ax = plt.subplots(figsize=(6, 4))
rng = np.random.default_rng(seed=42)
im = ax.imshow(rng.random((30, 30)), cmap="magma")
ax.set_title("Simulated activation map", fontsize=fonts["title"])
ax.set_xlabel("X", fontsize=fonts["axis_labels"])
ax.set_ylabel("Y", fontsize=fonts["axis_labels"])
ax.tick_params(labelsize=fonts["tick_labels"])
cbar = fig.colorbar(im, ax=ax, orientation="horizontal")
cbar.set_label("Values", fontsize=fonts["colorbar_title"])
cbar.ax.tick_params(labelsize=fonts["colorbar_ticks"])
plt.show()
print("-------------------------------")

### 3.2 `get_pyvista_fonts`
<a id="plot-get_pyvista_fonts"></a>

Get integer font sizes specifically for PyVista compatibility.

In [ ]:
########################### Testing get_pyvista_fonts ###########################
import clabtoolkit.plottools as cltplot

print("Test 1: Getting integer font sizes for an 8 x 6 inches PyVista rendering...")
print("NOTE: The debug information (_monitor_info) is excluded.")
pv_fonts = cltplot.get_pyvista_fonts(8, 6, screen_width=1920, screen_height=1080)
for key, value in pv_fonts.items():
    print(f"  {key:15s}: {value}")
print("-------------------------------")
print(" ")

print("Test 2: Comparing them with the float font sizes returned by calculate_font_sizes...")
fonts = cltplot.calculate_font_sizes(8, 6, screen_width=1920, screen_height=1080)
for key, value in pv_fonts.items():
    print(f"  {key:15s}: {fonts[key]} -> {value}")
print("-------------------------------")

---
## <a id="plot-other_methods"></a>4. Other Methods
Other auxiliary methods used to build the plots.

### 4.1 `generate_spherical_coords`
<a id="plot-generate_spherical_coords"></a>

Sample n_regions points uniformly inside a sphere of the given radius.

In [ ]:
########################### Testing generate_spherical_coords ###########################
import clabtoolkit.plottools as cltplot
import matplotlib.pyplot as plt
import numpy as np

print("Test 1: Sampling 5 points inside a sphere of radius 50...")
rng = np.random.default_rng(seed=42)
coords = cltplot.generate_spherical_coords(5, rng, radius=50.0)
print(f"  Shape: {coords.shape}")
print(np.round(coords, 2))
print("-------------------------------")
print(" ")

print("Test 2: Checking that all the points fall inside the sphere and that the sampling is reproducible...")
coords = cltplot.generate_spherical_coords(1000, np.random.default_rng(seed=1), radius=30.0)
distances = np.linalg.norm(coords, axis=1)
print(f"  Maximum distance to the center: {distances.max():.2f} (radius: 30.0)")
same = np.allclose(coords, cltplot.generate_spherical_coords(1000, np.random.default_rng(seed=1), radius=30.0))
print(f"  Same seed gives the same coordinates: {same}")
print("-------------------------------")
print(" ")

print("Test 3: Plotting a brain-like point cloud of 200 regions...")
coords = cltplot.generate_spherical_coords(200, np.random.default_rng(seed=42), radius=50.0)
fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(projection="3d")
ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=coords[:, 2], cmap="viridis", s=15)
ax.set_title("Simulated region centroids")
plt.show()
print("-------------------------------")